In [1]:
import math_toolkit
math_toolkit.notebook.activate()


# NamedExpression

Use a named expression when a formula should behave like one symbolic quantity until you ask for its definition.

`Symbol("A") @EqDef@ body`  
`Symbol("A")[i] @EqDef@ body`  
`IndexedBase("A")[i, j] @EqDef@ body`


In [2]:
Energy = Symbol("Energy") @ EqDef(latex=r"\mathcal{E}") @ ((p**2 + q**2) / 2)
energy_identity = Energy @Eq@ Energy
energy_plus_x = (Energy + x)

md("""The named expression displays as a symbol but expands when algebra needs the definition.

- Definition identity: {{ energy_identity }}
- Expanded expression inside a sum: {{ energy_plus_x }}
""")

The named expression displays as a symbol but expands when algebra needs the definition.

- Definition identity: $\mathcal{E} = \frac{p^{2}}{2} + \frac{q^{2}}{2}$
- Expanded expression inside a sum: $\frac{p^{2}}{2} + \frac{q^{2}}{2} + x$


## Core behavior

A named expression is an opaque SymPy expression with a remembered definition. It prints as the chosen name, composes with other expressions, and expands only when `` is requested.

Read `Symbol("A") @EqDef@ body` as "define the symbolic quantity `A` by the formula `body`." The result is the named expression, so assign it to the Python name you want to use later.


In [3]:
Cost = Symbol("Cost") @EqDef@ (p * x + q)
cost_expanded = (Cost + 1)

md("""Without display options, the named expression uses its symbol name until expansion.

- Named expression: {{ Cost }}
- Opaque sum: {{ Cost + 1 }}
- Expanded sum: {{ cost_expanded }}
""")

Without display options, the named expression uses its symbol name until expansion.

- Named expression: $\mathrm{Cost}$
- Opaque sum: $\mathrm{Cost} + 1$
- Expanded sum: $p x + q + 1$


The named expression itself can appear on either side of equations, inside larger expressions, or in substitutions. Expansion exposes the stored formula at the point where you want ordinary algebra to see it.


In [4]:
Balance = Symbol("Balance") @EqDef@ (p - q)
residual = Balance - x
expanded_residual = residual

md("""A named expression composes with ordinary algebra before expansion.

- Opaque residual: {{ residual }}
- Expanded residual: {{ expanded_residual }}
""")

A named expression composes with ordinary algebra before expansion.

- Opaque residual: $- x + \mathrm{Balance}$
- Expanded residual: $p - q - x$


## Common patterns

**Named formula boundaries.** Use a named expression when a formula has its own mathematical identity, such as an energy, residual, invariant, or loss.


In [5]:
Action = Symbol("Action") @ EqDef(latex=r"\mathcal{S}") @ (p * x - q)
action_identity = Action @Eq@ Action

md("""The LaTeX option changes the displayed head without changing the symbolic definition: {{ action_identity }}.""")

The LaTeX option changes the displayed head without changing the symbolic definition: $\mathcal{S} = p x - q$.

**Indexed families.** Put bracket placeholders in the left pattern when each label has its own expression. The same placeholder can be used in the defining body.


In [6]:
Term = Symbol("Term")[n] @EqDef@ (n**2 + 1)
term_identity = Term[k] @Eq@ Term[k]

md("""Indexed named expressions substitute the visible label into the definition: {{ term_identity }}.""")

Indexed named expressions substitute the visible label into the definition: $\mathrm{Term}_{k} = k^{2} + 1$.

**Multiple indexes.** Use an indexed base or an indexed symbol when the expression is naturally keyed by more than one label.


In [7]:
Kernel = IndexedBase("Kernel")[i, j] @ EqDef(latex=r"K") @ (i - j)
kernel_identity = Kernel[m, n] @Eq@ Kernel[m, n]

md("""Multi-index named expressions keep each structural label available to the definition: {{ kernel_identity }}.""")

Multi-index named expressions keep each structural label available to the definition: $K_{m, n} = m - n$.

## Options and Details

**Display templates.** The `latex=` option accepts the same template language used by `NamedFunction`: `#0` renders the normalized name, and `#{idx:1}` or `#{idx:*}` renders bracket labels.

In [8]:
AlphaEnergy = Symbol("alpha_energy")[n] @ EqDef(
    latex=r"\alpha_{#{idx:1}}^{#0}",
) @ (n + 1)
expanded_alpha_energy = AlphaEnergy[k]

md("""Display templates can include the printed atom name and the structural index.

- Displayed expression: {{ AlphaEnergy[k] }}
- Expanded definition: {{ expanded_alpha_energy }}
""")

Display templates can include the printed atom name and the structural index.

- Displayed expression: $\alpha_{k}^{\alpha_{energy}}$
- Expanded definition: $k + 1$


**Clearing definitions.** Use `None` on the right side to clear one stored definition. Clearing a specific indexed case can reveal the broader fallback; clearing the fallback leaves unmatched indexed expressions opaque.

In [9]:
SignalOption = Symbol("SignalOption") @ EqDef(indexed=True, latex=r"S") @ 0
SignalOption = SignalOption[i, j] @EqDef@ (i - j)
SignalOption = SignalOption[i, j] @EqDef@ None
cleared_two_index = SignalOption[k, m]

SignalOption = SignalOption @ EqDef(indexed=True) @ None
cleared_family = SignalOption[k]

md("""Assigning `None` clears definitions for matching indexed forms.

- Cleared two-index definition: {{ cleared_two_index }}
- Cleared family definition: {{ cleared_family }}
""")

Assigning `None` clears definitions for matching indexed forms.

- Cleared two-index definition: $0$
- Cleared family definition: $S_{k}$


## Semantics

A named expression has no call arguments. If the mathematical object should be applied to inputs, use `NamedFunction` or a function-left `EqDef` pattern such as `Function("F")(x) @EqDef@ body`.

Indexed named expressions dispatch by the number of bracket labels. A general indexed definition can handle any index count, and a more specific definition can override it for a particular shape.

In [ ]:
Signal = Symbol("Signal") @ EqDef(indexed=True, latex=r"S") @ 0
Signal = Signal[i, j] @EqDef@ (i - j)
signal_rows = [
    {"label": "one index", "definition": Signal[k] @Eq@ Signal[k]},
    {"label": "two indices", "definition": Signal[k, m] @Eq@ Signal[k, m]},
]

md("""More-specific indexed definitions override the broader family definition.

{table(header=["label", "definition"])}{{ signal_rows }}
""")

## Pitfalls

**The result must be assigned.** `EqDef` returns a new named expression or indexed family. It does not replace another Python variable just because the printed names match.


In [11]:
Symbol("Mass") @EqDef@ (p + q)

Mass = Symbol("Mass")
mass_without_binding = Mass

md("""A fresh symbol with the same printed name is not the registered named-expression object.

- Fresh symbol rewrite result: {{ mass_without_binding }}
""")

A fresh symbol with the same printed name is not the registered named-expression object.

- Fresh symbol rewrite result: $\mathrm{Mass}$


Assign the returned object when you want later code to use the definition.


In [12]:
Mass = Symbol("Mass") @EqDef@ (p + q)
mass_expanded = Mass

md("""Keeping the named-expression object preserves expansion: {{ mass_expanded }}.""")

Keeping the named-expression object preserves expansion: $p + q$.

**Index placeholders must be symbols.** The left pattern names placeholders, so create symbolic labels first and use those same labels in the body.


In [13]:
uPrime = Symbol("uPrime")
Mode = Symbol("Mode")[uPrime] @EqDef@ (uPrime + 1)
mode_identity = Mode[k] @Eq@ Mode[k]

md("""The definition uses the symbolic placeholder, so any later index can be substituted into it: {{ mode_identity }}.""")

The definition uses the symbolic placeholder, so any later index can be substituted into it: $\mathrm{Mode}_{k} = k + 1$.

**Use functions for inputs.** A named expression represents a quantity, not a callable rule. When you need `F(x)`, define a named function instead.


In [14]:
Potential = Function("Potential")(q) @ EqDef(latex=r"\mathcal{V}") @ (q**2 / 2)
potential_identity = Potential(q) @Eq@ Potential(q)

md("""A named expression can also use function-call notation when the displayed object should look like a potential: {{ potential_identity }}.""")

A named expression can also use function-call notation when the displayed object should look like a potential: $\mathcal{V}(q) = \frac{q^{2}}{2}$.

## See also

[NamedFunction](NamedFunction.ipynb)  
[Symbol](Symbol.ipynb)  
[Indexed](Indexed.ipynb)  
[Expression](Expression.ipynb)  
[rewrite](rewrite.ipynb)  
[Function families](../tutorials/function_families.ipynb)  
[Composing structured expressions](../tutorials/composing_expressions.ipynb)
